#  Lesson 6: Professional NLP Pipelines
In this lesson, we move from manual counting (N-grams) to **Industrial-strength NLP pipelines**. We will compare the two most popular libraries used in Linguistics and AI today:

1.  **spaCy**: Extremely fast, efficient, and easy to use for production.
2.  **Stanza**: Created by Stanford NLP. It is slower but provides state-of-the-art accuracy using Deep Learning and supports more linguistic features (like Dependency Parsing).

We will analyze James Joyce's *Finnegans Wake*, a notoriously difficult text, to see how these models perform.

###  1. Environment Setup (Colab Version)
When working with large texts and Deep Learning models (like Stanza), we need more power than a standard laptop often provides. 
- **GPU Acceleration**: We are using Google Colab to access a GPU (Graphics Processing Unit) which makes the neural networks run 10x-100x faster.
- **Google Drive**: We mount our drive to save and load large files (like our processed corpora).

In [ ]:
import spacy
from google.colab import drive
drive.mount('/gdrive')
import os
import requests
from bs4 import BeautifulSoup
os.chdir("/gdrive/My Drive/CattolicaColabExperiment")


###  2. Data Acquisition: Finnegans Wake
We load our text. *Finnegans Wake* is an ideal test case for NLP because of its linguistic complexity, multilingual puns, and non-standard syntax.

In [ ]:
finn = open('finneganswake.txt').read()
type(finn)

##  Deep Dive: The Architecture of spaCy

To use spaCy effectively, you must understand its internal logic. Unlike many libraries that provide a collection of independent tools, spaCy is built around a unified **Pipeline**.

### 1. The `nlp` Object
The `nlp` object is the engine of spaCy. When you call `nlp = spacy.load("en_core_web_sm")`, you are loading:
- **Weights**: The binary data that allows the model to make predictions.
- **The Pipeline**: A sequence of components (functions) that the text passes through.
- **The Vocabulary**: A `Vocab` object that stores shared data (like strings and word vectors) to save memory.

### 2. The Container Objects: `Doc`, `Token`, and `Span`
When you process text with `doc = nlp("Hello world")`, spaCy creates three main types of objects:
1.  **`Doc`**: The container for the whole sequence. It holds all the information extracted from the text.
2.  **`Token`**: Represents an individual linguistic unit (a word, a punctuation mark, a whitespace). 
3.  **`Span`**: A slice of a `Doc`. For example, an entity like "James Joyce" is a `Span` containing two tokens.

### 3. The Processing Pipeline
When you call `nlp(text)`, the following steps happen in order:
1.  **Tokenizer**: Turns raw text into a `Doc` object. (This is the only part that cannot be disabled).
2.  **Tagger**: Assigns Part-of-Speech tags.
3.  **Lemmatizer**: Assigns base forms to words.
4.  **Parser**: Computes the syntactic dependencies (how words relate to each other).
5.  **NER (Entity Recognizer)**: Identifies and labels named entities.

### 4. Model Design: Statistical vs. Transformer
- **`sm` (Small)**: These are efficient **Convolutional Neural Networks (CNN)**. They don't include word vectors but use context to guess tags. They are fast and run on any CPU.
- **`trf` (Transformer)**: These use architectures like BERT. They are significantly more accurate but require a GPU and much more memory.

###  3. Linguistic Analysis with spaCy
spaCy uses highly optimized models to perform multiple tasks at once:
1.  **Tokenization**: Splitting into words.
2.  **Lemmatization**: Finding the root form (e.g., *saw* -> *see*).
3.  **POS Tagging**: Identifying Nouns, Verbs, etc.
4.  **NER**: Finding People, Places, and Organizations.

In [ ]:
# import spacy

# Download the English model (if not already present)
# !python -m spacy download en_core_web_sm

# Load the model
# Increase the limit to 2 million characters

nlp = spacy.load("en_core_web_sm")

We process the first and last 100,000 characters to compare different parts of the book.

In [ ]:
doc1 = nlp(finn[:100000])
doc2 = nlp(finn[-100000:])

####  Named Entity Comparison
We check if the *types* of entities found in the beginning match the types found at the end. In a complex book like this, the 'semantic landscape' might shift significantly.

In [ ]:
ners1 == ners2

###  4. Processing the Full Text
Now we run the pipeline on the *entire* book. Note how long it takes. 
> **Pro Tip**: Processing a whole book takes time. We use `pickle` to save the resulting `Doc` object to a file so we don't have to run the NLP pipeline again every time we open the notebook.

In [ ]:
import time
s = time.time()
nlp.max_length = 2000000
finndoc = nlp(finn)
e = time.time()
print(e-s)

In [ ]:
import pickle
pickle.dump(finndoc, open("finndoc.pkl", "wb"))


###  5. Exporting to CoNLL-U Format
In Linguistics, the **CoNLL-U** format is a tab-separated standard for representing annotated text. Each line is a token with its properties (Lemma, POS, Dependency head, etc.). This allows our data to be used in other specialized linguistics software.

In [ ]:

import spacy

nlp = spacy.load("en_core_web_sm")

def to_conll(doc):
    """Converts a spaCy doc to a CoNLL-U formatted string."""
    rows = []
    for i, token in enumerate(doc, 1):
        # CoNLL-U Columns: ID, FORM, LEMMA, UPOS, XPOS, FEATS, HEAD, DEPREL, DEPS, MISC

        # Determine the head ID (0 for root, otherwise the 1-based index)
        head = 0 if token.head == token else token.head.i + 1

        # NER info in the MISC column
        ner = f"Entity={token.ent_type_}" if token.ent_type_ else "_"

        row = [
            str(i),                         # ID
            token.text,                     # FORM
            token.lemma_,                    # LEMMA
            token.pos_,                      # UPOS
            token.tag_,                      # XPOS
            "_",                             # FEATS (can be complex, using _ for simplicity)
            str(head),                      # HEAD
            token.dep_,                      # DEPREL
            "_",                             # DEPS
            ner                              # MISC
        ]
        rows.append("\t".join(row))
    return "\n".join(rows)


with open("finn.conllu", "w", encoding="utf-8") as f:

    f.write(to_conll(finndoc))


print("Done! Check output.conllu in the Colab file browser.")

###  6. Deep Learning Analysis with Stanza
**Stanza** is built on top of PyTorch and uses sophisticated Neural Networks. It is often more accurate than spaCy for complex syntax but requires a GPU to run efficiently.

In [ ]:
!pip install stanza
from stanza import Pipeline
nlp_stanza = Pipeline('en', processors='tokenize,pos,depparse,lemma,ner')


Compare the time taken by Stanza vs spaCy. You'll see that accuracy comes at a computational cost!

In [ ]:

import time
s = time.time()
finndocstz = nlp_stanza(finn)
e = time.time()
print(e-s)

In [ ]:
from stanza.utils.conll import CoNLL

CoNLL.write_doc2conll(finndocstz, 'finn_stanza.conll')

##  Technical Spotlight: Stanza's Architecture

While spaCy is built for speed and production, **Stanza** (developed by Stanford NLP) is built for linguistic precision.

### How Stanza Works:
1.  **Neural from the Ground Up**: Stanza is built on **PyTorch**. Every component (from tokenization to dependency parsing) is a deep neural network.
2.  **Universal Dependencies (UD)**: Stanza is the gold standard for UD. It is designed to work consistently across 70+ languages using the same set of linguistic labels.
3.  **Character-Level Language Models**: Unlike spaCy's small models which look at word shapes, Stanza uses a **Character-level model**. This allows it to handle extremely complex morphology and "Out-of-Vocabulary" words (words the model hasn't seen before) by looking at their internal structure (prefixes, suffixes, roots).
4.  **Bi-LSTM & Attention**: Stanza uses Bidirectional Long Short-Term Memory (Bi-LSTM) networks and Attention mechanisms to capture the long-distance relationships between words in a sentence.

**Summary**: Use **spaCy** when you need to process millions of documents quickly. Use **Stanza** when you are doing deep linguistic research where every grammatical label must be as accurate as possible.